# Additional Graphs – India Cricket Analytics

This notebook adds 6 new visualizations exploring batting, bowling, format comparisons, and player career stats.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── consistent style ──────────────────────────────────────────────────────────
INDIA_BLUE  = '#003366'
INDIA_ORANGE = '#FF6B00'
INDIA_GREEN = '#138808'
PALETTE_3   = [INDIA_BLUE, INDIA_ORANGE, INDIA_GREEN]

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.bbox': 'tight'})

# ── load processed data ───────────────────────────────────────────────────────
kpis     = pd.read_csv('../data/processed/match_kpis.csv')
outcomes = pd.read_csv('../data/processed/match_outcomes.csv')

# ── load career data ──────────────────────────────────────────────────────────
bat_odi  = pd.read_csv('../data/raw/espn_india_odi_batting_career.csv')
bat_test = pd.read_csv('../data/raw/espn_india_test_batting_career.csv')
bat_t20  = pd.read_csv('../data/raw/espn_india_t20_batting_career.csv')
bowl_odi = pd.read_csv('../data/raw/espn_india_odi_bowling_career.csv')

print('KPI rows:', len(kpis), '| Outcome rows:', len(outcomes))

## Graph 1 – Win Rate by Format and Venue Type

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

win_fmt_venue = (
    kpis.groupby(['format', 'venue_type'])['win_loss_flag']
    .mean()
    .mul(100)
    .reset_index(name='win_pct')
)

formats  = win_fmt_venue['format'].unique()
venues   = ['Home', 'Away']
x        = np.arange(len(formats))
width    = 0.35

colors = {vt: c for vt, c in zip(venues, [INDIA_BLUE, INDIA_ORANGE])}

for i, vt in enumerate(venues):
    sub = win_fmt_venue[win_fmt_venue['venue_type'] == vt].set_index('format').reindex(formats)
    bars = ax.bar(x + (i - 0.5) * width, sub['win_pct'], width,
                  label=vt, color=colors[vt], edgecolor='white', linewidth=0.8)
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax.text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(formats, fontsize=12)
ax.set_ylabel('Win Percentage (%)')
ax.set_title('India Win Rate by Format and Venue Type', fontsize=14, fontweight='bold', pad=12)
ax.set_ylim(0, 85)
ax.legend(title='Venue', framealpha=0.8)
ax.axhline(50, color='grey', lw=0.8, ls='--', alpha=0.6)
ax.text(2.6, 51.5, '50% line', color='grey', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/figures/fig8_win_rate_format_venue.png')
plt.show()
print('Saved fig8_win_rate_format_venue.png')

## Graph 2 – Distribution of India's Batting Strike Rate by Format (Violin)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

order = ['Test', 'ODI', 'T20I']
sub   = kpis.dropna(subset=['india_sr'])

parts = ax.violinplot(
    [sub[sub['format'] == fmt]['india_sr'].values for fmt in order],
    positions=range(len(order)),
    showmedians=True, showextrema=True
)

for i, (pc, col) in enumerate(zip(parts['bodies'], PALETTE_3)):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
parts['cmedians'].set_color('white')
parts['cmedians'].set_linewidth(2)

ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, fontsize=12)
ax.set_ylabel('Batting Strike Rate')
ax.set_title("Distribution of India's Batting Strike Rate by Format",
             fontsize=14, fontweight='bold', pad=12)

patches = [mpatches.Patch(facecolor=c, alpha=0.7, label=fmt)
           for fmt, c in zip(order, PALETTE_3)]
ax.legend(handles=patches, loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig('../reports/figures/fig9_sr_distribution_violin.png')
plt.show()
print('Saved fig9_sr_distribution_violin.png')

## Graph 3 – Bowling Economy vs Wickets (Scatter with Win/Loss colour)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, fmt in zip(axes, ['ODI', 'T20I', 'Test']):
    sub = kpis[(kpis['format'] == fmt)].dropna(subset=['india_bowl_econ', 'india_bowl_wkts'])
    wins  = sub[sub['win_loss_flag'] == 1]
    losses = sub[sub['win_loss_flag'] == 0]

    ax.scatter(losses['india_bowl_econ'], losses['india_bowl_wkts'],
               color=INDIA_ORANGE, alpha=0.5, s=25, label='Loss', zorder=2)
    ax.scatter(wins['india_bowl_econ'], wins['india_bowl_wkts'],
               color=INDIA_BLUE, alpha=0.6, s=25, label='Win', zorder=3)

    # trend line for wins
    if len(wins) > 5:
        m, b = np.polyfit(wins['india_bowl_econ'], wins['india_bowl_wkts'], 1)
        xs = np.linspace(wins['india_bowl_econ'].min(), wins['india_bowl_econ'].max(), 50)
        ax.plot(xs, m*xs + b, color=INDIA_BLUE, lw=1.5, ls='--', alpha=0.8)

    ax.set_title(fmt, fontsize=13, fontweight='bold')
    ax.set_xlabel('Bowling Economy Rate')
    if ax == axes[0]:
        ax.set_ylabel('Wickets Taken')
    ax.legend(fontsize=8, framealpha=0.7)

fig.suptitle('Bowling Economy vs Wickets Taken – Win/Loss', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/fig10_bowling_economy_scatter.png')
plt.show()
print('Saved fig10_bowling_economy_scatter.png')

## Graph 4 – Top 10 ODI Batters: Runs vs Average (Bubble = Matches)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

top = bat_odi.dropna(subset=['Runs', 'Ave', 'Mat']).nlargest(10, 'Runs')
top['Ave']  = pd.to_numeric(top['Ave'],  errors='coerce')
top['Mat']  = pd.to_numeric(top['Mat'],  errors='coerce')
top['Runs'] = pd.to_numeric(top['Runs'], errors='coerce')
top = top.dropna(subset=['Ave'])

cmap   = plt.cm.get_cmap('Blues')
norm   = plt.Normalize(top['Mat'].min(), top['Mat'].max())
colors = [cmap(norm(m)) for m in top['Mat']]

sc = ax.scatter(top['Ave'], top['Runs'], s=top['Mat'] * 1.8,
                c=top['Mat'], cmap='Blues', edgecolors=INDIA_BLUE,
                linewidth=0.8, alpha=0.85, zorder=3)

for _, row in top.iterrows():
    name = row['Player'].split(' ')[-1]   # surname only
    ax.annotate(name, (row['Ave'], row['Runs']),
                textcoords='offset points', xytext=(6, 4),
                fontsize=9, color=INDIA_BLUE)

cb = plt.colorbar(sc, ax=ax, shrink=0.7)
cb.set_label('Matches Played', fontsize=9)

ax.set_xlabel('Batting Average', fontsize=12)
ax.set_ylabel('Total Runs', fontsize=12)
ax.set_title('Top 10 India ODI Batters – Runs vs Average\n(bubble size = matches played)',
             fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('../reports/figures/fig11_top10_odi_batters_bubble.png')
plt.show()
print('Saved fig11_top10_odi_batters_bubble.png')

## Graph 5 – India Win Rate Trend Over Time (rolling 20-match)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=False)

for ax, fmt, color in zip(axes, ['ODI', 'T20I', 'Test'], PALETTE_3):
    sub = (
        kpis[kpis['format'] == fmt]
        .sort_values('start_date')
        [['start_date', 'win_loss_flag']]
        .dropna()
    )
    sub['start_date'] = pd.to_datetime(sub['start_date'])
    sub = sub.set_index('start_date').sort_index()
    roll = sub['win_loss_flag'].rolling(window=20, min_periods=5).mean().mul(100)

    ax.fill_between(roll.index, roll, alpha=0.15, color=color)
    ax.plot(roll.index, roll, color=color, lw=2)
    ax.axhline(50, color='grey', lw=0.8, ls='--', alpha=0.7)
    ax.set_ylabel('Win % (20-match rolling)', fontsize=9)
    ax.set_title(f'{fmt} – Rolling Win Rate', fontsize=11, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('India Rolling Win Rate Over Time (20-match window)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/fig12_rolling_win_rate_trend.png')
plt.show()
print('Saved fig12_rolling_win_rate_trend.png')

## Graph 6 – Batting First vs Second: Win Rate Heatmap by Era & Format

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, bat_first_val, title in zip(
    axes,
    [1, 0],
    ['Batting First', 'Chasing']
):
    sub = kpis[kpis['india_bat_first'] == bat_first_val]
    pivot = (
        sub.groupby(['era_label', 'format'])['win_loss_flag']
        .mean()
        .mul(100)
        .unstack('format')
        .reindex(['Early 2000s', 'Transition Era', 'Modern Era'])
        [['Test', 'ODI', 'T20I']]
    )

    sns.heatmap(
        pivot, ax=ax, annot=True, fmt='.1f', cmap='YlOrRd',
        linewidths=0.5, linecolor='white',
        vmin=25, vmax=85,
        annot_kws={'fontsize': 11, 'fontweight': 'bold'},
        cbar_kws={'label': 'Win %'}
    )
    ax.set_title(f'India Win % – {title}', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('')
    ax.set_ylabel('Era')
    ax.tick_params(axis='x', rotation=0)
    ax.tick_params(axis='y', rotation=0)

fig.suptitle('Win % Heatmap: Batting First vs Chasing – by Era & Format',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('../reports/figures/fig13_bat_first_chase_heatmap.png')
plt.show()
print('Saved fig13_bat_first_chase_heatmap.png')

---
All 6 new figures saved to `reports/figures/`.